
## Final Retail Analytics Dashboard Dataset:

Now combine the transformations to answer real business questions.

In [0]:
%run ./10_Statistical_Analysis


### KPI Summary:

**Calculate the main business metrics for delivered orders.**

In [0]:
retail_kpi_df = delivered_sales_df.agg(
    F.sum("net_revenue").alias("total_revenue"),
    F.countDistinct("order_id").alias("total_orders"),
    F.countDistinct("customer_id").alias("total_customers"),
    F.sum("quantity").alias("total_units_sold"),
    F.countDistinct("product_id").alias("unique_products_sold"),
    F.avg("net_revenue").alias("average_item_revenue")
)

display(retail_kpi_df)


### Business questions answered:

**- What is the total delivered sales revenue?
- How many delivered orders were placed?
- How many customers purchased products?
- How many units were sold?
- What is the average revenue per order item?**

### Note that average_item_revenue is the average per item row, not the average order value.


### Top Five Products by Revenue:

In [0]:
top_products_df = delivered_sales_df.groupBy(
    "product_id",
    "product_name",
    "category"
).agg(
    F.sum("quantity").alias("total_units_sold"),
    F.sum("net_revenue").alias("total_revenue")
).orderBy(
    F.desc("total_revenue")
).limit(5)

display(top_products_df)


**Business use: Identify the products generating the highest delivered sales revenue.**



### Store Performance Ranking:

In [0]:
store_performance_df = delivered_sales_df.groupBy(
    "store_id",
    "store_name",
    "store_city"
).agg(
    F.sum("net_revenue").alias("total_revenue"),
    F.countDistinct("order_id").alias("total_orders")
)

store_rank_window = Window.orderBy(
    F.desc("total_revenue")
)

store_performance_df = store_performance_df.withColumn(
    "store_rank",
    F.dense_rank().over(store_rank_window)
)

display(store_performance_df)


**Business use: Compare stores and identify the strongest-performing locations.**


**Customer Segmentation:**

In [0]:
customer_segmentation_df = customer_summary_df.withColumn(
    "spending_segment",
    F.when(
        F.col("total_spending") >= 50000,
        "High Value"
    ).when(
        F.col("total_spending") >= 10000,
        "Medium Value"
    ).otherwise(
        "Low Value"
    )
)

display(
    customer_segmentation_df.orderBy(
        F.desc("total_spending")
    )
)


### Monthly Category Pivot:

In [0]:
final_monthly_dashboard_df = delivered_sales_df.groupBy(
    "order_month"
).pivot(
    "category",
    ["Electronics", "Accessories", "Furniture"]
).agg(
    F.sum("net_revenue")
).fillna(0)

display(final_monthly_dashboard_df.orderBy("order_month"))


# Save the Final Dataset to Delta Lake:


### Save to a Databricks volume:

In [0]:
output_path = "/Volumes/workspace/default/data_volume/retail_sales/"

retail_wide_df.write \
    .format("delta") \
    .mode("overwrite") \
    .save(output_path)


### Read the Delta dataset:

In [0]:
retail_sales_df = spark.read.format("delta").load(
    output_path
)

display(retail_sales_df)


## Data Quality Validation:

A data engineer should validate the output before making it available to business users.


### Check for Null Values:

In [0]:
sales_df.select(
    *[
        F.sum(
            F.col(column).isNull().cast("int")
        ).alias(column + "_nulls")
        for column in sales_df.columns
    ]
).display()


### Check Duplicate Order Items:

In [0]:
duplicate_items_df = order_items_df.groupBy(
    "item_id"
).count().filter(
    F.col("count") > 1
)

display(duplicate_items_df)

### Check Invalid Revenue:

In [0]:
invalid_revenue_df = sales_df.filter(
    F.col("net_revenue") < 0
)

display(invalid_revenue_df)


### Reconcile Row Counts:

In [0]:
print("Customers:", customers_df.count())
print("Orders:", orders_clean_df.count())
print("Order items:", order_items_df.count())
print("Products:", products_df.count())
print("Stores:", stores_df.count())
print("Payments:", payments_df.count())
print("Final sales rows:", sales_df.count())


### Save Your PySpark DataFrames as Tables:

In [0]:
# Main sales detail table
sales_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.retail_sales_detail")

# Delivered sales table
delivered_sales_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.retail_delivered_sales")

# Customer summary
customer_summary_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.retail_customer_summary")

# Monthly sales
monthly_sales_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.retail_monthly_sales")

# Store performance
store_performance_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.retail_store_performance")

print("Retail analytics tables created successfully.")

In [0]:
%sql
SHOW TABLES IN workspace.default;


### View the table data:

**For example, to view the first 10 rows of your sales detail table:**

In [0]:
%sql
SELECT *
FROM workspace.default.retail_sales_detail;

**To verify the delivered sales table:**

In [0]:
%sql
SELECT *
FROM workspace.default.retail_delivered_sales
LIMIT 10;


**To count the records:**

In [0]:
%sql
SELECT COUNT(*) AS total_records
FROM workspace.default.retail_sales_detail;